# Churn Analysis and Retention

> **Sample data disclaimer:** This notebook uses sample data, not the original business data. Churn rates, model metrics, and customer profiles are specific to this sample and may differ when original data is used.


In [281]:
import sys
sys.path.append("..")

In [282]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import unicodedata
from random import sample

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_score

# Fix for newer SciPy versions: scikitplot expects scipy.interp, which was removed
import scipy
if not hasattr(scipy, "interp"):
    from scipy.interpolate import interp1d
    scipy.interp = interp1d

from scikitplot.metrics import plot_lift_curve

from pathlib import Path


# Read the Data


## Products


In [283]:
path = "../data/sample/"
output_dir = Path("../charts_and_figures/churn_recommendation")
output_dir.mkdir(exist_ok=True)

In [284]:
from src.data import load_raw_tables
customers_df, products_df, sales_df = load_raw_tables(path)

# Exploratory Data Analysis and Feature Engineering


In [285]:
sales_by_order = sales_df.groupby(['SALE_ID', 'SALE_DATE', 'CUSTOMER_ID', 'SALE_CATEGORY']).agg(
    PRODUCT_COUNT=('PRODUCT_ID', 'nunique'),
    SALE_COUNT=('SALE_ID', 'nunique'),
    TOTAL_QUANTITY=('QUANTITY', 'sum'),
).reset_index()
print('Shape:', sales_by_order.shape)
sales_by_order.head()

Shape: (60000, 7)


,SALE_ID,SALE_DATE,CUSTOMER_ID,SALE_CATEGORY,PRODUCT_COUNT,SALE_COUNT,TOTAL_QUANTITY
0,1,2020/01/02,2044,STORE,1,1,1
1,2,2020/01/02,4458,STORE,4,1,5
2,3,2020/01/02,4450,GROOMING,3,1,3
3,4,2020/01/02,4381,GROOMING,1,1,1
4,5,2020/01/02,1244,STORE,1,1,1


In [286]:
customer_sales = customers_df.merge(sales_by_order, how='left', on='CUSTOMER_ID')
print('Shape:', customer_sales.shape)
customer_sales.head()

Shape: (60002, 10)


,CUSTOMER_ID,SEX,BIRTH_DATE,FIRST_PURCHASE_DATE,SALE_ID,SALE_DATE,SALE_CATEGORY,PRODUCT_COUNT,SALE_COUNT,TOTAL_QUANTITY
0,1,<NA>,1961/10/09,2018/05/17,814.0,2020/01/11,STORE,1.0,1.0,1
1,1,<NA>,1961/10/09,2018/05/17,5881.0,2020/03/10,STORE,2.0,1.0,3
2,1,<NA>,1961/10/09,2018/05/17,17493.0,2020/07/17,STORE,1.0,1.0,1
3,1,<NA>,1961/10/09,2018/05/17,51154.0,2021/07/18,STORE,2.0,1.0,3
4,2,<NA>,1955/07/13,2017/06/28,4333.0,2020/02/21,STORE,4.0,1.0,5


In [287]:
print('Customers WITHOUT sales:', customer_sales[customer_sales.SALE_ID.isnull()].CUSTOMER_ID.nunique())
print('Customers WITH sales:', customer_sales[customer_sales.SALE_ID.notnull()].CUSTOMER_ID.nunique())

Customers WITHOUT sales: 2
Customers WITH sales: 4998


**Customers with no sales will be removed from the modeling dataset.**


In [288]:
customer_sales = customer_sales[customer_sales.SALE_ID.notnull()]
print('Shape:', customer_sales.shape)
customer_sales.head()

Shape: (60000, 10)


,CUSTOMER_ID,SEX,BIRTH_DATE,FIRST_PURCHASE_DATE,SALE_ID,SALE_DATE,SALE_CATEGORY,PRODUCT_COUNT,SALE_COUNT,TOTAL_QUANTITY
0,1,<NA>,1961/10/09,2018/05/17,814.0,2020/01/11,STORE,1.0,1.0,1
1,1,<NA>,1961/10/09,2018/05/17,5881.0,2020/03/10,STORE,2.0,1.0,3
2,1,<NA>,1961/10/09,2018/05/17,17493.0,2020/07/17,STORE,1.0,1.0,1
3,1,<NA>,1961/10/09,2018/05/17,51154.0,2021/07/18,STORE,2.0,1.0,3
4,2,<NA>,1955/07/13,2017/06/28,4333.0,2020/02/21,STORE,4.0,1.0,5


### Number of Sales per Customer


In [289]:
sales_per_customer = customer_sales.groupby('CUSTOMER_ID')[['SALE_ID']].count().reset_index()
sales_per_customer = sales_per_customer.rename(columns={'SALE_ID':'SALE_COUNT'}, inplace=False)
print(sales_per_customer.shape)
sales_per_customer.head()

(4998, 2)


,CUSTOMER_ID,SALE_COUNT
0,1,4
1,2,11
2,3,9
3,4,2
4,5,10


In [290]:
sales_per_customer.SALE_COUNT.value_counts().head(15)

SALE_COUNT
6     481
7     464
8     436
9     435
5     428
4     339
10    306
11    269
12    223
13    194
3     167
14    155
16    111
15     97
2      89
Name: count, dtype: int64

### Time Between Purchases


In [291]:
print('Last purchase:', pd.to_datetime(customer_sales.SALE_DATE.max()))
reference_date = pd.to_datetime(customer_sales.SALE_DATE.max())

Last purchase: 2021-10-26 00:00:00


In [292]:
customer_sales["SALE_DATE"] = pd.to_datetime(
    customer_sales["SALE_DATE"],
    errors="coerce"
)



In [293]:
customer_sales = customer_sales.sort_values(['CUSTOMER_ID', 'SALE_DATE'])
customer_sales = customer_sales[customer_sales['SALE_DATE'] <= reference_date]
print('Shape:', customer_sales.shape)
customer_sales['NEXT_SALE_DATE'] = customer_sales.groupby('CUSTOMER_ID')['SALE_DATE'].shift(-1)
customer_sales['NEXT_SALE_DATE'] = customer_sales['NEXT_SALE_DATE'].fillna(reference_date)
customer_sales.head()

Shape: (60000, 10)


,CUSTOMER_ID,SEX,BIRTH_DATE,FIRST_PURCHASE_DATE,SALE_ID,SALE_DATE,SALE_CATEGORY,PRODUCT_COUNT,SALE_COUNT,TOTAL_QUANTITY,NEXT_SALE_DATE
0,1,<NA>,1961/10/09,2018/05/17,814.0,2020-01-11,STORE,1.0,1.0,1,2020-03-10
1,1,<NA>,1961/10/09,2018/05/17,5881.0,2020-03-10,STORE,2.0,1.0,3,2020-07-17
2,1,<NA>,1961/10/09,2018/05/17,17493.0,2020-07-17,STORE,1.0,1.0,1,2021-07-18
3,1,<NA>,1961/10/09,2018/05/17,51154.0,2021-07-18,STORE,2.0,1.0,3,2021-10-26
4,2,<NA>,1955/07/13,2017/06/28,4333.0,2020-02-21,STORE,4.0,1.0,5,2020-03-25


In [294]:
customer_sales['DAYS_BETWEEN_SALES'] = (customer_sales['NEXT_SALE_DATE'] - customer_sales['SALE_DATE']).dt.days
customer_sales.head()

,CUSTOMER_ID,SEX,BIRTH_DATE,FIRST_PURCHASE_DATE,SALE_ID,SALE_DATE,SALE_CATEGORY,PRODUCT_COUNT,SALE_COUNT,TOTAL_QUANTITY,NEXT_SALE_DATE,DAYS_BETWEEN_SALES
0,1,<NA>,1961/10/09,2018/05/17,814.0,2020-01-11,STORE,1.0,1.0,1,2020-03-10,59
1,1,<NA>,1961/10/09,2018/05/17,5881.0,2020-03-10,STORE,2.0,1.0,3,2020-07-17,129
2,1,<NA>,1961/10/09,2018/05/17,17493.0,2020-07-17,STORE,1.0,1.0,1,2021-07-18,366
3,1,<NA>,1961/10/09,2018/05/17,51154.0,2021-07-18,STORE,2.0,1.0,3,2021-10-26,100
4,2,<NA>,1955/07/13,2017/06/28,4333.0,2020-02-21,STORE,4.0,1.0,5,2020-03-25,33


In [295]:
days_between_sales = customer_sales.groupby('DAYS_BETWEEN_SALES')[['SALE_ID']].count().reset_index()
days_between_sales = days_between_sales.rename(columns={'SALE_ID':'SALE_COUNT'}, inplace=False)
print(days_between_sales.shape)
days_between_sales.head(15)

(494, 2)


,DAYS_BETWEEN_SALES,SALE_COUNT
0,0,1232
1,1,2270
2,2,2055
3,3,1827
4,4,1757
5,5,1608
6,6,1459
7,7,1339
8,8,1299
9,9,1286


# Build the Modeling Dataset


In [296]:
customer_sales.SALE_DATE.dt.strftime('%Y%m').sort_values(ascending=False).value_counts(sort=False)

SALE_DATE
202110    2225
202109    2645
202108    2701
202107    2996
202106    2857
202105    2657
202104    2636
202103    2813
202102    2492
202101    2759
202012    3664
202011    2683
202010    2733
202009    2582
202008    2708
202007    2945
202006    2844
202005    2710
202004    2583
202003    2662
202002    2506
202001    2599
Name: count, dtype: int64

## Example Month


### Set the Reference Date


In [297]:
reference_month = '202107'
churn_days = 90
reference_date = pd.to_datetime(reference_month, format='%Y%m')
response_end_date = reference_date + timedelta(days=churn_days)
customer_window_start = reference_date + timedelta(days=-churn_days)
print('Reference date:', reference_date)
print('Response window ends:', response_end_date)
print('Customer window starts:', customer_window_start)

Reference date: 2021-07-01 00:00:00
Response window ends: 2021-09-29 00:00:00
Customer window starts: 2021-04-02 00:00:00


In [298]:
customer_sales.head()

,CUSTOMER_ID,SEX,BIRTH_DATE,FIRST_PURCHASE_DATE,SALE_ID,SALE_DATE,SALE_CATEGORY,PRODUCT_COUNT,SALE_COUNT,TOTAL_QUANTITY,NEXT_SALE_DATE,DAYS_BETWEEN_SALES
0,1,<NA>,1961/10/09,2018/05/17,814.0,2020-01-11,STORE,1.0,1.0,1,2020-03-10,59
1,1,<NA>,1961/10/09,2018/05/17,5881.0,2020-03-10,STORE,2.0,1.0,3,2020-07-17,129
2,1,<NA>,1961/10/09,2018/05/17,17493.0,2020-07-17,STORE,1.0,1.0,1,2021-07-18,366
3,1,<NA>,1961/10/09,2018/05/17,51154.0,2021-07-18,STORE,2.0,1.0,3,2021-10-26,100
4,2,<NA>,1955/07/13,2017/06/28,4333.0,2020-02-21,STORE,4.0,1.0,5,2020-03-25,33


### Select the customer base (customers who purchased in the 90 days before the reference date)


In [299]:
# Select the customer base (customers who purchased in the 90 days before the reference date)
eligible_customers = customer_sales[(customer_sales['SALE_DATE'] >= customer_window_start) &
                                     (customer_sales['SALE_DATE'] < reference_date)]\
    .groupby('CUSTOMER_ID')[['SALE_DATE']].max().reset_index()
eligible_customers = eligible_customers.rename(columns={'SALE_DATE':'LAST_SALE_DATE'})
print('Eligible customer count:', eligible_customers.shape)
eligible_customers.head()

Eligible customer count: (3490, 2)


,CUSTOMER_ID,LAST_SALE_DATE
0,3,2021-05-24
1,5,2021-06-08
2,6,2021-06-27
3,7,2021-06-04
4,8,2021-06-25


### Select purchases in the 90 days after the reference date to label churn


In [300]:
# Select purchases in the 90 days after the reference date to label churn
next_purchase_by_customer = customer_sales[(customer_sales['SALE_DATE'] >= reference_date) &
                                           (customer_sales['SALE_DATE'] < response_end_date)].\
    groupby('CUSTOMER_ID')[['SALE_DATE']].min().reset_index()
next_purchase_by_customer = next_purchase_by_customer.rename(columns={'SALE_DATE':'NEXT_SALE_DATE'})
next_purchase_by_customer['REFERENCE_DATE'] = reference_date
next_purchase_by_customer['NEXT_MONTH_DATE'] = next_purchase_by_customer['REFERENCE_DATE'] + pd.DateOffset(months=1)
next_month_boundary = next_purchase_by_customer['NEXT_MONTH_DATE'].max()
print('Next-month date:', next_month_boundary)
next_purchase_by_customer['ADJUSTED_NEXT_SALE_DATE'] = next_purchase_by_customer[['NEXT_SALE_DATE','NEXT_MONTH_DATE']].min(axis=1)
print('Shape:', next_purchase_by_customer.shape)
next_purchase_by_customer.head()

Next-month date: 2021-08-01 00:00:00
Shape: (3526, 5)


,CUSTOMER_ID,NEXT_SALE_DATE,REFERENCE_DATE,NEXT_MONTH_DATE,ADJUSTED_NEXT_SALE_DATE
0,1,2021-07-18,2021-07-01,2021-08-01,2021-07-18
1,2,2021-07-30,2021-07-01,2021-08-01,2021-07-30
2,7,2021-07-01,2021-07-01,2021-08-01,2021-07-01
3,9,2021-08-24,2021-07-01,2021-08-01,2021-08-01
4,10,2021-07-23,2021-07-01,2021-08-01,2021-07-23


### Label churn based on the time between the last purchase before the reference date and the first subsequent purchase


In [301]:
# Label churn based on the time between the last purchase before the reference date and the first subsequent purchase
customer_churn_labels = eligible_customers.merge(next_purchase_by_customer, on='CUSTOMER_ID', how='left')
print('Shape:', customer_churn_labels.shape)
customer_churn_labels['DAYS_TO_NEXT_SALE'] = (customer_churn_labels['NEXT_SALE_DATE'] - customer_churn_labels['LAST_SALE_DATE']).dt.days
customer_churn_labels['ADJUSTED_NEXT_SALE_DATE'] = customer_churn_labels['ADJUSTED_NEXT_SALE_DATE'].fillna(next_month_boundary)
customer_churn_labels['ADJUSTED_DAYS_TO_NEXT_SALE'] = (customer_churn_labels['ADJUSTED_NEXT_SALE_DATE'] - customer_churn_labels['LAST_SALE_DATE']).dt.days
customer_churn_labels['CHURN_FLAG'] = np.where(customer_churn_labels['DAYS_TO_NEXT_SALE'] < churn_days, 0, 1)
customer_churn_labels['MONTHLY_CHURN_FLAG'] = np.where(customer_churn_labels['ADJUSTED_DAYS_TO_NEXT_SALE'] < churn_days, 0, 1)
print(customer_churn_labels.CHURN_FLAG.value_counts())
customer_churn_labels.head()

Shape: (3490, 6)
CHURN_FLAG
0    1988
1    1502
Name: count, dtype: int64


,CUSTOMER_ID,LAST_SALE_DATE,NEXT_SALE_DATE,REFERENCE_DATE,NEXT_MONTH_DATE,ADJUSTED_NEXT_SALE_DATE,DAYS_TO_NEXT_SALE,ADJUSTED_DAYS_TO_NEXT_SALE,CHURN_FLAG,MONTHLY_CHURN_FLAG
0,3,2021-05-24,NaT,NaT,NaT,2021-08-01,NaN,69,1,0
1,5,2021-06-08,NaT,NaT,NaT,2021-08-01,NaN,54,1,0
2,6,2021-06-27,NaT,NaT,NaT,2021-08-01,NaN,35,1,0
3,7,2021-06-04,2021-07-01,2021-07-01,2021-08-01,2021-07-01,27.0,27,0,0
4,8,2021-06-25,NaT,NaT,NaT,2021-08-01,NaN,37,1,0


### Create features using only sales before the reference date


In [302]:
# Create features using only sales before the reference date
feature_sales = customer_sales[(customer_sales['SALE_DATE'] < reference_date) & 
                                   (customer_sales['CUSTOMER_ID'].isin(eligible_customers.CUSTOMER_ID.unique().tolist()))]
print('Shape:', feature_sales.shape)
feature_sales.head()

Shape: (41239, 12)


,CUSTOMER_ID,SEX,BIRTH_DATE,FIRST_PURCHASE_DATE,SALE_ID,SALE_DATE,SALE_CATEGORY,PRODUCT_COUNT,SALE_COUNT,TOTAL_QUANTITY,NEXT_SALE_DATE,DAYS_BETWEEN_SALES
15,3,<NA>,1999/08/12,2018/04/22,11060.0,2020-05-09,STORE,1.0,1.0,1,2020-07-04,56
16,3,<NA>,1999/08/12,2018/04/22,16210.0,2020-07-04,STORE,2.0,1.0,2,2020-08-17,44
17,3,<NA>,1999/08/12,2018/04/22,20271.0,2020-08-17,STORE,2.0,1.0,2,2020-09-08,22
18,3,<NA>,1999/08/12,2018/04/22,22175.0,2020-09-08,STORE,2.0,1.0,2,2021-02-15,160
19,3,<NA>,1999/08/12,2018/04/22,37246.0,2021-02-15,STORE,1.0,1.0,1,2021-03-14,27


In [303]:
# Aggregate by customer to create features
customer_features = feature_sales.groupby(['CUSTOMER_ID']).agg({'SALE_COUNT':'sum',
                                                               'PRODUCT_COUNT':'sum',
                                                               'TOTAL_QUANTITY':'sum',
                                                               'SALE_DATE':['min', 'max'],
                                                               'BIRTH_DATE':'max'}).reset_index()
customer_features.columns = ['CUSTOMER_ID', 'SALE_COUNT', 'PRODUCT_COUNT', 'TOTAL_QUANTITY',
                            'FIRST_SALE_DATE', 'LAST_SALE_DATE', 'BIRTH_DATE']
print(customer_features.shape)
customer_features.head()

(3490, 7)


,CUSTOMER_ID,SALE_COUNT,PRODUCT_COUNT,TOTAL_QUANTITY,FIRST_SALE_DATE,LAST_SALE_DATE,BIRTH_DATE
0,3,9.0,17.0,17,2020-05-09,2021-05-24,1999/08/12
1,5,10.0,22.0,25,2020-02-16,2021-06-08,1964/01/07
2,6,4.0,8.0,9,2020-02-09,2021-06-27,1980/08/08
3,7,11.0,21.0,29,2020-01-05,2021-06-04,1977/12/19
4,8,4.0,6.0,9,2020-02-20,2021-06-25,1955/05/08


In [304]:
# Remove accents from category names
def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

# Pivot to get sales counts for each category
category_features = pd.pivot_table(feature_sales, values='SALE_COUNT', index='CUSTOMER_ID', columns='SALE_CATEGORY', aggfunc='sum', fill_value=0).astype(int)
category_features.columns = ['CATEGORY_'+strip_accents(col.replace('-', '')).upper() for col in category_features.columns]
category_features = category_features.reset_index()
print(category_features.shape)
category_features.head()

(3490, 6)


,CUSTOMER_ID,CATEGORY_CLINIC,CATEGORY_DIGITAL,CATEGORY_ECOMMERCE,CATEGORY_GROOMING,CATEGORY_STORE
0,3,0,0,0,0,9
1,5,2,0,0,1,7
2,6,0,0,0,0,4
3,7,0,0,0,0,11
4,8,0,0,1,0,3


In [305]:
customer_model_features = customer_features.merge(category_features, on='CUSTOMER_ID', how='left')
print(customer_model_features.shape)
customer_model_features.head()

(3490, 12)


,CUSTOMER_ID,SALE_COUNT,PRODUCT_COUNT,TOTAL_QUANTITY,FIRST_SALE_DATE,LAST_SALE_DATE,BIRTH_DATE,CATEGORY_CLINIC,CATEGORY_DIGITAL,CATEGORY_ECOMMERCE,CATEGORY_GROOMING,CATEGORY_STORE
0,3,9.0,17.0,17,2020-05-09,2021-05-24,1999/08/12,0,0,0,0,9
1,5,10.0,22.0,25,2020-02-16,2021-06-08,1964/01/07,2,0,0,1,7
2,6,4.0,8.0,9,2020-02-09,2021-06-27,1980/08/08,0,0,0,0,4
3,7,11.0,21.0,29,2020-01-05,2021-06-04,1977/12/19,0,0,0,0,11
4,8,4.0,6.0,9,2020-02-20,2021-06-25,1955/05/08,0,0,1,0,3


In [306]:
date_columns = ["BIRTH_DATE", "FIRST_SALE_DATE", "LAST_SALE_DATE"]

for column in date_columns:
    customer_model_features[column] = pd.to_datetime(
        customer_model_features[column],
        errors="coerce"
    )

reference_date = pd.Timestamp(reference_date)
customer_model_features["REFERENCE_DATE"] = reference_date

In [307]:
# Calcula demais features
customer_model_features['REFERENCE_DATE'] = reference_date
customer_model_features['AGE'] = customer_model_features['REFERENCE_DATE'].dt.year - customer_model_features['BIRTH_DATE'].dt.year
customer_model_features['CUSTOMER_TENURE_DAYS'] = (customer_model_features['REFERENCE_DATE'] - customer_model_features['FIRST_SALE_DATE']) / np.timedelta64(1, 'D')
customer_model_features['DAYS_BETWEEN_SALES'] = (customer_model_features['LAST_SALE_DATE'] - customer_model_features['FIRST_SALE_DATE']) / np.timedelta64(1, 'D')
customer_model_features['PURCHASE_FREQUENCY_DAYS'] = (customer_model_features['DAYS_BETWEEN_SALES'] / customer_model_features['SALE_COUNT'])
customer_model_features['RECENCY_DAYS'] = (customer_model_features['REFERENCE_DATE'] - customer_model_features['LAST_SALE_DATE']) / np.timedelta64(1, 'D')
customer_model_features = customer_model_features.drop(['FIRST_SALE_DATE', 'LAST_SALE_DATE', 'BIRTH_DATE', 'AGE'], axis=1)
print(customer_model_features.shape)
customer_model_features.head()

(3490, 14)


,CUSTOMER_ID,SALE_COUNT,PRODUCT_COUNT,TOTAL_QUANTITY,CATEGORY_CLINIC,CATEGORY_DIGITAL,CATEGORY_ECOMMERCE,CATEGORY_GROOMING,CATEGORY_STORE,REFERENCE_DATE,CUSTOMER_TENURE_DAYS,DAYS_BETWEEN_SALES,PURCHASE_FREQUENCY_DAYS,RECENCY_DAYS
0,3,9.0,17.0,17,0,0,0,0,9,2021-07-01,418.0,380.0,42.222222,38.0
1,5,10.0,22.0,25,2,0,0,1,7,2021-07-01,501.0,478.0,47.800000,23.0
2,6,4.0,8.0,9,0,0,0,0,4,2021-07-01,508.0,504.0,126.000000,4.0
3,7,11.0,21.0,29,0,0,0,0,11,2021-07-01,543.0,516.0,46.909091,27.0
4,8,4.0,6.0,9,0,0,1,0,3,2021-07-01,497.0,491.0,122.750000,6.0


In [308]:
# Join the final modeling dataset
modeling_data = customer_model_features.merge(customer_churn_labels[['CUSTOMER_ID', 'CHURN_FLAG', 'MONTHLY_CHURN_FLAG']], on='CUSTOMER_ID', how='left')
print(modeling_data.shape)
modeling_data.head()

(3490, 16)


,CUSTOMER_ID,SALE_COUNT,PRODUCT_COUNT,TOTAL_QUANTITY,CATEGORY_CLINIC,CATEGORY_DIGITAL,CATEGORY_ECOMMERCE,CATEGORY_GROOMING,CATEGORY_STORE,REFERENCE_DATE,CUSTOMER_TENURE_DAYS,DAYS_BETWEEN_SALES,PURCHASE_FREQUENCY_DAYS,RECENCY_DAYS,CHURN_FLAG,MONTHLY_CHURN_FLAG
0,3,9.0,17.0,17,0,0,0,0,9,2021-07-01,418.0,380.0,42.222222,38.0,1,0
1,5,10.0,22.0,25,2,0,0,1,7,2021-07-01,501.0,478.0,47.800000,23.0,1,0
2,6,4.0,8.0,9,0,0,0,0,4,2021-07-01,508.0,504.0,126.000000,4.0,1,0
3,7,11.0,21.0,29,0,0,0,0,11,2021-07-01,543.0,516.0,46.909091,27.0,0,0
4,8,4.0,6.0,9,0,0,1,0,3,2021-07-01,497.0,491.0,122.750000,6.0,1,0


## Build the Dataset Across Multiple Reference Dates


In [309]:
reference_months = ['202101', '202102', '202103', '202104', '202105', '202106', '202107']
churn_days = 90
print(reference_months)

['202101', '202102', '202103', '202104', '202105', '202106', '202107']


In [ ]:
def build_modeling_dataset(reference_month):
    reference_date = pd.to_datetime(reference_month, format='%Y%m').normalize()
    response_end_date = reference_date + pd.Timedelta(days=churn_days)
    customer_window_start = reference_date - pd.Timedelta(days=churn_days)

    sales = customer_sales.copy()
    sales['SALE_DATE'] = pd.to_datetime(sales['SALE_DATE'], errors='coerce')
    sales['BIRTH_DATE'] = pd.to_datetime(sales['BIRTH_DATE'], errors='coerce')
    sales = sales.dropna(subset=['SALE_DATE', 'CUSTOMER_ID', 'SALE_ID'])
    if sales.empty:
        raise ValueError('No valid sales are available to build the cohort.')

    # The outcome is only valid when the complete follow-up window is observed.
    required_through = response_end_date - pd.Timedelta(days=1)
    observed_through = sales['SALE_DATE'].max().normalize()
    if observed_through < required_through:
        raise ValueError(
            f'Incomplete follow-up for {reference_date.date()}: expected data through '
            f'{required_through.date()}, observed through {observed_through.date()}.'
        )

    eligible_sales = sales.loc[
        sales['SALE_DATE'].ge(customer_window_start)
        & sales['SALE_DATE'].lt(reference_date)
    ]
    eligible_customers = eligible_sales.groupby('CUSTOMER_ID', as_index=False).agg(
        LAST_SALE_DATE=('SALE_DATE', 'max')
    )
    if eligible_customers.empty:
        raise ValueError(f'No eligible customers for {reference_date.date()}.')

    future_sales = sales.loc[
        sales['SALE_DATE'].ge(reference_date)
        & sales['SALE_DATE'].lt(response_end_date)
    ]
    next_purchase = future_sales.groupby('CUSTOMER_ID')['SALE_DATE'].min()
    customer_churn_labels = eligible_customers.merge(
        next_purchase.rename('NEXT_SALE_DATE'), on='CUSTOMER_ID', how='left'
    )
    customer_churn_labels['DAYS_TO_NEXT_SALE'] = (
        customer_churn_labels['NEXT_SALE_DATE']
        - customer_churn_labels['LAST_SALE_DATE']
    ).dt.days
    monthly_boundary = reference_date + pd.DateOffset(months=1)
    customer_churn_labels['ADJUSTED_NEXT_SALE_DATE'] = (
        customer_churn_labels['NEXT_SALE_DATE'].fillna(monthly_boundary)
    ).clip(upper=monthly_boundary)
    customer_churn_labels['ADJUSTED_DAYS_TO_NEXT_SALE'] = (
        customer_churn_labels['ADJUSTED_NEXT_SALE_DATE']
        - customer_churn_labels['LAST_SALE_DATE']
    ).dt.days
    customer_churn_labels['CHURN_FLAG'] = (
        customer_churn_labels['DAYS_TO_NEXT_SALE'].ge(churn_days)
        | customer_churn_labels['DAYS_TO_NEXT_SALE'].isna()
    ).astype(int)
    customer_churn_labels['MONTHLY_CHURN_FLAG'] = (
        customer_churn_labels['ADJUSTED_DAYS_TO_NEXT_SALE'].ge(churn_days)
        .astype(int)
    )

    # Features use all available history strictly before the reference date.
    feature_sales = sales.loc[
        sales['SALE_DATE'].lt(reference_date)
        & sales['CUSTOMER_ID'].isin(eligible_customers['CUSTOMER_ID'])
    ].copy()
    customer_features = feature_sales.groupby('CUSTOMER_ID').agg(
        SALE_COUNT=('SALE_ID', 'nunique'),
        PRODUCT_COUNT=('PRODUCT_COUNT', 'sum'),
        TOTAL_QUANTITY=('TOTAL_QUANTITY', 'sum'),
        FIRST_SALE_DATE=('SALE_DATE', 'min'),
        LAST_SALE_DATE=('SALE_DATE', 'max'),
    ).reset_index()

    category_names = {
        category: 'CATEGORY_' + strip_accents(category.replace('-', '')).upper()
        for category in sales['SALE_CATEGORY'].dropna().unique()
    }
    category_counts = (
        feature_sales.dropna(subset=['SALE_CATEGORY'])
        .groupby(['CUSTOMER_ID', 'SALE_CATEGORY'])['SALE_COUNT']
        .sum().unstack(fill_value=0)
        .rename(columns=category_names)
        .reindex(columns=list(category_names.values()), fill_value=0)
    )
    customer_features = customer_features.merge(
        category_counts, left_on='CUSTOMER_ID', right_index=True, how='left'
    )
    category_columns = list(category_names.values())
    customer_features[category_columns] = customer_features[category_columns].fillna(0)
    customer_features['REFERENCE_DATE'] = reference_date
    customer_features['CUSTOMER_TENURE_DAYS'] = (
        customer_features['REFERENCE_DATE'] - customer_features['FIRST_SALE_DATE']
    ).dt.days
    customer_features['DAYS_BETWEEN_SALES'] = (
        customer_features['LAST_SALE_DATE'] - customer_features['FIRST_SALE_DATE']
    ).dt.days
    interval_count = customer_features['SALE_COUNT'].sub(1)
    customer_features['PURCHASE_FREQUENCY_DAYS'] = (
        customer_features['DAYS_BETWEEN_SALES']
        .div(interval_count.where(interval_count.gt(0)))
        .fillna(0)
    )
    customer_features['RECENCY_DAYS'] = (
        customer_features['REFERENCE_DATE'] - customer_features['LAST_SALE_DATE']
    ).dt.days
    customer_features = customer_features.drop(
        columns=['FIRST_SALE_DATE', 'LAST_SALE_DATE']
    )

    modeling_data = customer_features.merge(
        customer_churn_labels[
            ['CUSTOMER_ID', 'CHURN_FLAG', 'MONTHLY_CHURN_FLAG']
        ],
        on='CUSTOMER_ID', how='left', validate='one_to_one'
    )
    return modeling_data

In [ ]:
modeling_datasets = []

for reference_month in reference_months:
    
    modeling_data = build_modeling_dataset(reference_month)
    modeling_datasets.append(modeling_data)
    
modeling_data_all = pd.concat(modeling_datasets)
print(modeling_data_all.shape)
modeling_data_all.head()

In [ ]:
modeling_data_all.isna().sum()

In [ ]:
modeling_data_all['CATEGORY_ECOMMERCE'] = modeling_data_all['CATEGORY_ECOMMERCE'].fillna(0).astype(int)

In [ ]:
modeling_data_all.info()

In [ ]:
modeling_columns = modeling_data_all.columns
print(modeling_columns)

## Churn in the Modeling Dataset


### Full Dataset: Unrestricted Churn Label


In [ ]:
pd.pivot_table(modeling_data_all, values='CUSTOMER_ID', index='REFERENCE_DATE', columns='CHURN_FLAG', aggfunc='count', fill_value=0).astype(int)

### Full Dataset: Monthly Churn Label


In [ ]:
pd.pivot_table(modeling_data_all, values='CUSTOMER_ID', index='REFERENCE_DATE', columns='MONTHLY_CHURN_FLAG', aggfunc='count', fill_value=0).astype(int)

### Customers with Recency >= 60: Unrestricted Churn Label


In [ ]:
pd.pivot_table(modeling_data_all[modeling_data_all['RECENCY_DAYS'] >= 60],
               values='CUSTOMER_ID', index='REFERENCE_DATE', columns='CHURN_FLAG', aggfunc='count', fill_value=0).astype(int)

### Customers with Recency >= 60: Monthly Churn Label


In [ ]:
pd.pivot_table(modeling_data_all[modeling_data_all['RECENCY_DAYS'] >= 60],
               values='CUSTOMER_ID', index='REFERENCE_DATE', columns='MONTHLY_CHURN_FLAG', aggfunc='count', fill_value=0).astype(int)

In [ ]:
modeling_data_all = modeling_data_all.set_index(['CUSTOMER_ID', 'REFERENCE_DATE'])
modeling_data_all.head()

In [ ]:
modeling_data_all = modeling_data_all[modeling_data_all['RECENCY_DAYS'] >= 60]
print(modeling_data_all.shape)
modeling_data_all.head()

# Modeling


In [ ]:
X = modeling_data_all.drop(['CHURN_FLAG', 'MONTHLY_CHURN_FLAG'], axis=1)
print(X.shape)
X.head()

In [ ]:
y = modeling_data_all[['CHURN_FLAG']]
print(y.shape)
y.head()

## Train-Test Split


In [ ]:
# Set a random_seed for reproducible results
random_seed = 10

# Use train_test_split to create the test set
features_training_all, features_test, labels_training_all, labels_test = train_test_split(X, y, random_state=random_seed, test_size=0.25, stratify=y)

print("Test set X", features_test.shape)
print("Test set y", labels_test.shape)

In [ ]:
# Repeat the split to create a validation set from the remaining training data
features_train, features_validation, labels_train, labels_validation = train_test_split(features_training_all, labels_training_all, random_state=random_seed,
                                                  test_size=0.33, stratify=labels_training_all)

print("Train set X", features_train.shape)
print("Train set y", labels_train.shape)
print("Validation set X", features_validation.shape)
print("Validation set y", labels_validation.shape)

## Models


In [ ]:
def plot_feature_importances(model):
    # Listando features
    n_features = X.shape[1]
    # Horizontal bar chart of feature importance
    plt.barh(range(n_features), model.feature_importances_, align='center')
    # Add tick marks to the y-axis
    plt.yticks(np.arange(n_features), X.columns)
    plt.xlabel("Feature importance")
    plt.ylabel("Feature")
    plt.ylim(-1, n_features)

### Decision Tree


In [ ]:
# Define candidate hyperparameter values
parameter_grid = {'max_depth': [2, 3, 4, 5, 6, 7, 8, 9],
          'min_samples_split': [10, 15, 20],
          'min_samples_leaf': [3, 5, 10]}

# Create a DecisionTreeClassifier
decision_tree = DecisionTreeClassifier(random_state=random_seed)

# Create a StratifiedKFold cross-validation splitter
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_seed)

# Create a GridSearchCV object with the selected parameter grid
grid_search = GridSearchCV(decision_tree, param_grid=parameter_grid, scoring='roc_auc',
                           cv=skf.split(features_training_all, labels_training_all))

# Train the model with grid search
grid_search.fit(features_training_all, labels_training_all)

# Print the best hyperparameter combination
print('Best hyperparameters:')
print(grid_search.best_params_)

In [ ]:
# Create a decision_tree with the best hyperparameters
decision_tree = DecisionTreeClassifier(random_state=random_seed, max_depth=grid_search.best_params_['max_depth'],
                              min_samples_split=grid_search.best_params_['min_samples_split'],
                              min_samples_leaf=grid_search.best_params_['min_samples_leaf'])

# Fit the model on the training dataset
decision_tree.fit(features_training_all, labels_training_all)

# Predict the validation set
tree_validation_predictions = decision_tree.predict(features_validation)
tree_validation_probabilities = [x[1] for x in decision_tree.predict_proba(features_validation)]

# Calculate and print evaluation metrics
print("Training accuracy: {:.3f}".format(decision_tree.score(features_train, labels_train)))
print("Validation accuracy: {:.3f}".format(decision_tree.score(features_validation, labels_validation)))
print(classification_report(labels_validation, tree_validation_predictions))
print(confusion_matrix(labels_validation, tree_validation_predictions))
print("ROC AUC: {:.3f}".format(roc_auc_score(labels_validation, tree_validation_probabilities)))

# Plot feature importance
plot_feature_importances(decision_tree)

In [ ]:
%matplotlib inline

tree_test_roc_auc = roc_auc_score(labels_test, [x[1] for x in decision_tree.predict_proba(features_test)])
fpr, tpr, thresholds = roc_curve(labels_test, decision_tree.predict_proba(features_test)[:,1])
plt.figure()
plt.plot(fpr, tpr, label='Decision Tree (AUC = %0.2f)' % tree_test_roc_auc)
plt.plot([0, 1], [0, 1],'r--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()

### XGBoost


In [ ]:
# Define candidate hyperparameter values
parameter_grid = {'learning_rate': [0.1, 0.2, 0.3],
          'alpha': [5, 10, 15],
          'colsample_bytree': [0.1, 0.3, 0.5, 0.7, 1.0],
          'max_depth': [3, 4, 5]}

# Create the model
xgb_model_cv_gs = xgb.XGBClassifier(objective="binary:logistic", random_state=random_seed,
                                    eval_metric="auc", n_estimators=10,use_label_encoder=False)

# Create grid search with stratified cross-validation
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_seed)
grid_search = GridSearchCV(xgb_model_cv_gs, param_grid=parameter_grid, scoring='roc_auc',
                           cv=skf.split(features_training_all, labels_training_all))

# Train the model and print the best hyperparameters
grid_search.fit(features_training_all, labels_training_all)
print('\n Best hyperparameters:')
print(grid_search.best_params_)

# Get cross-validation results
cv_results = pd.DataFrame(grid_search.cv_results_)
print("Mean validation accuracy: {:.3f} +/- {:.3f}".format(
  cv_results[cv_results.rank_test_score == 1].mean_test_score.values[0],
  cv_results[cv_results.rank_test_score == 1].std_test_score.values[0])
)

# Set the best hyperparameter values
xgb_model_cv_gs.set_params(learning_rate = grid_search.best_params_['learning_rate'],
                           alpha = grid_search.best_params_['alpha'],
                           colsample_bytree = grid_search.best_params_['colsample_bytree'],
                           max_depth = grid_search.best_params_['max_depth'])

# Train the model using the best hyperparameters
xgb_model_cv_gs.fit(features_training_all, labels_training_all)

# Plot feature importance
xgb.plot_importance(xgb_model_cv_gs)
plt.show()

In [ ]:
# Predict the validation set
xgb_validation_predictions = decision_tree.predict(features_validation)
xgb_validation_probabilities = [x[1] for x in decision_tree.predict_proba(features_validation)]

# Calculate and print evaluation metrics
print("Training accuracy: {:.3f}".format(xgb_model_cv_gs.score(features_train, labels_train)))
print("Validation accuracy: {:.3f}".format(xgb_model_cv_gs.score(features_validation, labels_validation)))
print(classification_report(labels_validation, xgb_validation_predictions))
print(confusion_matrix(labels_validation, xgb_validation_predictions))
print("ROC AUC: {:.3f}".format(roc_auc_score(labels_validation, xgb_validation_probabilities)))

In [ ]:
%matplotlib inline

xgb_test_roc_auc = roc_auc_score(labels_test, [x[1] for x in xgb_model_cv_gs.predict_proba(features_test)])
fpr, tpr, thresholds = roc_curve(labels_test, xgb_model_cv_gs.predict_proba(features_test)[:,1])
plt.figure()
plt.plot(fpr, tpr, label='XGBoost (AUC = %0.2f)' % xgb_test_roc_auc)
plt.plot([0, 1], [0, 1],'r--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()

## Model Evaluation


### AUC


In [ ]:
print("Decision Tree AUC: {:.3f}".format(xgb_test_roc_auc))
print("XGBoost AUC: {:.3f}".format(tree_test_roc_auc))
print("Decision Tree AUC: {:.3f}".format(xgb_test_roc_auc))
print("XGBoost AUC: {:.3f}".format(tree_test_roc_auc))

### Precision


In [ ]:
# Generate predictions for the test dataset
tree_test_predictions = decision_tree.predict(features_test)
xgb_test_predictions = xgb_model_cv_gs.predict(features_test)

# Print each model's precision
print(precision_score(labels_test, tree_test_predictions))
print(precision_score(labels_test, xgb_test_predictions))

### Lift


In [ ]:
tree_test_probabilities = decision_tree.predict_proba(features_test)
xgb_test_probabilities = xgb_model_cv_gs.predict_proba(features_test)

plot_lift_curve(labels_test, tree_test_probabilities)
plot_lift_curve(labels_test, xgb_test_probabilities)
plt.show()

## Holdout Dataset


In [ ]:
holdout_reference_month = '202107'

In [ ]:
holdout_data = build_modeling_dataset(holdout_reference_month)[modeling_columns].set_index(['CUSTOMER_ID', 'REFERENCE_DATE'])
holdout_data.head()

In [ ]:
holdout_data.CHURN_FLAG.value_counts()

In [ ]:
holdout_data.MONTHLY_CHURN_FLAG.value_counts()

In [ ]:
holdout_data = holdout_data[holdout_data['RECENCY_DAYS'] >= 60]
print(holdout_data.shape)
holdout_data.head()

In [ ]:
holdout_data.CHURN_FLAG.value_counts()

In [ ]:
holdout_data.MONTHLY_CHURN_FLAG.value_counts()

In [ ]:
features_holdout = holdout_data.drop(['CHURN_FLAG', 'MONTHLY_CHURN_FLAG'], axis=1)
print(features_holdout.shape)
features_holdout.head()

In [ ]:
labels_holdout = holdout_data[['CHURN_FLAG']]
print(labels_holdout.shape)
labels_holdout.head()

### AUC


In [ ]:
# Generate predictions for the test dataset
tree_holdout_probabilities = [x[1] for x in decision_tree.predict_proba(features_holdout)]
xgb_holdout_probabilities = [x[1] for x in xgb_model_cv_gs.predict_proba(features_holdout)]

print("Decision Tree AUC: {:.3f}".format(roc_auc_score(labels_holdout, tree_holdout_probabilities)))
print("XGBoost AUC: {:.3f}".format(roc_auc_score(labels_holdout, xgb_holdout_probabilities)))

### Lift


In [ ]:
tree_holdout_predictions = decision_tree.predict_proba(features_holdout)
xgb_holdout_predictions = xgb_model_cv_gs.predict_proba(features_holdout)

plot_lift_curve(labels_holdout, tree_holdout_predictions)
plot_lift_curve(labels_holdout, xgb_holdout_predictions)
plt.show()

### Lift Data


In [ ]:
ax = plot_lift_curve(labels_holdout, tree_holdout_predictions)

In [ ]:
lift_lines = []
lines = ax.get_lines()[0:-1]
for line in lines:
    xd = line.get_xdata()
    yd = line.get_ydata()
    lift_lines.append(pd.DataFrame({'y':yd}, index=xd))

In [ ]:
lift_data = pd.concat(lift_lines, axis=1)
print(lift_data.shape)
lift_data.head()

In [ ]:
lift_data.to_csv('./Dados_Apresentacao/lift_churn.csv', sep=';', decimal=',')